## NB 1



## Goal

Understand the network connection data and test whether we can identify
connections labelled as attacks.

## What we do

1. Explore the dataset and its labels.
2. Try a simple rule: flag connections with no reply.
3. Train Isolation Forest and Random Forest models.
4. Compare predictions with the correct answers.
5. Investigate false alarms and explain the model’s limitations.

## Main finding

The selected Random Forest caught about 99% of attacks on the official
test set, but it also flagged 11,544 normal connections. These false
alarms are the main limitation of the current detector.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("C:/Projects/21_Cyber_Threat_Detection/data/UNSW_NB15_training-set.csv")
df = pd.read_csv(data_path)

print("Rows and columns:", df.shape)
display(df[["dur", "proto", "spkts", "dpkts", "attack_cat", "label"]].head())

Rows and columns: (175341, 45)


,dur,proto,spkts,dpkts,attack_cat,label
0,0.121478,tcp,6,4,Normal,0
1,0.649902,tcp,14,38,Normal,0
2,1.623129,tcp,8,16,Normal,0
3,1.681642,tcp,12,12,Normal,0
4,0.449454,tcp,10,6,Normal,0


In [2]:
# Count how many normal and attack rows 
print(df["label"].value_counts())
print(df["attack_cat"].value_counts())

label
1    119341
0     56000
Name: count, dtype: int64
attack_cat
Normal            56000
Generic           40000
Exploits          33393
Fuzzers           18184
DoS               12264
Reconnaissance    10491
Analysis           2000
Backdoor           1746
Shellcode          1133
Worms               130
Name: count, dtype: int64


In [3]:
# Meet one normal row and one attack row side by side
columns = ["dur", "proto", "spkts", "dpkts", "sbytes", "dbytes", "attack_cat", "label"]

normal = df[df["label"] == 0].iloc[0]
attack = df[df["label"] == 1].iloc[0]

comparison = pd.DataFrame({
    "Normal example": normal[columns],
    "Attack example": attack[columns]
})

display(comparison)

,Normal example,Attack example
dur,0.121478,0.000009
proto,tcp,ddp
spkts,6,2
dpkts,4,0
sbytes,258,200
dbytes,172,0
attack_cat,Normal,Backdoor
label,0,1


In [4]:
# Check whether "no reply" is common acorss all normal and attack rows
df.groupby("label")["dpkts"].apply(
    lambda packets: (packets == 0).mean() * 100
).round(1)

label
0    11.9
1    65.0
Name: dpkts, dtype: float64

In [5]:
# Which attack types has no reply
no_reply_by_type = (
    df.groupby("attack_cat")["dpkts"]
    .apply(lambda packets: (packets == 0).mean() * 100)
    .sort_values(ascending=False)
    .round(1)
)
display(no_reply_by_type)

attack_cat
Generic           98.7
Backdoor          81.2
DoS               79.5
Analysis          71.4
Reconnaissance    51.0
Shellcode         50.8
Exploits          39.5
Fuzzers           35.2
Normal            11.9
Worms             11.5
Name: dpkts, dtype: float64

In [6]:
## Flag a row if it got no reply
flagged = df["dpkts"] == 0

pd.crosstab(
    df["label"],
    flagged,
    rownames=["Actual label (0=normal, 1=attack)"],
    colnames=["Flagged for no reply"]
)

Flagged for no reply,False,True
"Actual label (0=normal, 1=attack)",,
0,49344,6656
1,41715,77626


In [7]:
# Put a score on simple rule
from sklearn.metrics import precision_score, recall_score

print("Precision:", round(precision_score(df["label"], flagged), 3))
print("Recall:", round(recall_score(df["label"], flagged), 3))

Precision: 0.921
Recall: 0.65


In [8]:
# Check whether the data has missing values
missing = df.isna().sum()
display(missing[missing > 0].sort_values(ascending=False))

Series([], dtype: int64)

In [9]:
# check some numeric columns have infinity
import numpy as np
numeric = df.select_dtypes(include="number")
infinity_counts = np.isinf(numeric).sum()
display(infinity_counts[infinity_counts > 0])

Series([], dtype: int64)

In [ ]:
# Checks whether columns contains words rather than numbers
text_columns = df.select_dtypes(include="object").columns
print(text_columns.tolist())['proto', 'service', 'state', 'attack_cat']

['proto', 'service', 'state', 'attack_cat']


In [11]:
display(df["proto"].value_counts().head(10))

proto
tcp       79946
udp       63283
unas      12084
arp        2859
ospf       2595
sctp       1150
any         300
gre         225
ipv6        201
mobile      201
Name: count, dtype: int64

In [12]:
# Prepare the clues X and the answer y
X = df.drop(columns=["id", "attack_cat", "label"])
y = df["label"]

print("Clues:", X.shape)
print("Answers:", y.shape)

Clues: (175341, 42)
Answers: (175341,)


In [13]:
# Create Yes/No column for each value
X_encoded = pd.get_dummies(
    X,
    columns=["proto", "service", "state"],
    dtype="int8"
)

print("Before:", X.shape)
print("After:", X_encoded.shape)
display(X_encoded.head(2))

Before: (175341, 42)
After: (175341, 194)


,dur,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,...,service_ssl,state_CON,state_ECO,state_FIN,state_INT,state_PAR,state_REQ,state_RST,state_URN,state_no
0,0.121478,6,4,258,172,74.087490,252,254,14158.942380,8495.365234,...,0,0,0,1,0,0,0,0,0,0
1,0.649902,14,38,734,42014,78.473372,62,252,8395.112305,503571.312500,...,0,0,0,1,0,0,0,0,0,0


In [14]:
# Teach anomaly model what normla connections look like
from sklearn.ensemble import IsolationForest
X_normal = X_encoded[y == 0]
print("Normal examples for training:", X_normal.shape)

model = IsolationForest(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
model.fit(X_normal)

Normal examples for training: (56000, 194)


IsolationForest(n_jobs=-1, random_state=42)

In [15]:
model_flags = (model.predict(X_encoded) == -1).astype(int)

pd.crosstab(
    y,
    model_flags,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["Model flagged (0=no, 1=yes)"]
)

"Model flagged (0=no, 1=yes)",0,1
"Actual (0=normal, 1=attack)",,
0,52057,3943
1,72111,47230


In [16]:
# Test the model on connections it hasn't seen before
test_df = pd.read_csv("../data/UNSW_NB15_testing-set.csv")

test_X = test_df.drop(columns=["id", "attack_cat", "label"])
test_X = pd.get_dummies(
    test_X,
    columns=["proto", "service", "state"],
    dtype="int8"
)

# Give the test data the same clue columns, in the same order, as training
test_X = test_X.reindex(columns=X_encoded.columns, fill_value=0)

test_y = test_df["label"]
test_flags = (model.predict(test_X) == -1).astype(int)

print("Test rows:", len(test_df))
display(pd.crosstab(
    test_y,
    test_flags,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["Model flagged (0=no, 1=yes)"]
))

Test rows: 82332


"Model flagged (0=no, 1=yes)",0,1
"Actual (0=normal, 1=attack)",,
0,33378,3622
1,28929,16403


In [17]:
# Check whether no reply rule
simple_test_flags = (test_df["dpkts"] == 0).astype(int)

display(pd.crosstab(
    test_y,
    simple_test_flags,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["No-reply rule flagged (0=no, 1=yes)"]
))

"No-reply rule flagged (0=no, 1=yes)",0,1
"Actual (0=normal, 1=attack)",,
0,30807,6193
1,15519,29813


In [18]:
print("Actual test labels:")
print(test_y.value_counts())

print("\nModel confusion matrix:")
print(pd.crosstab(test_y, test_flags))

Actual test labels:
label
1    45332
0    37000
Name: count, dtype: int64

Model confusion matrix:
col_0      0      1
label              
0      33378   3622
1      28929  16403


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_encoded,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Validation labels:")
print(y_val.value_counts())

Training: (131505, 194)
Validation: (43836, 194)
Validation labels:
label
1    29836
0    14000
Name: count, dtype: int64


In [20]:
# Retrain the Isolation Forest using only the normal row from the training portion
from sklearn.ensemble import IsolationForest

normal_train = X_train[y_train == 0]
print("Normal rows used to teach the model:", len(normal_train))

val_model = IsolationForest(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
val_model.fit(normal_train)

Normal rows used to teach the model: 42000


IsolationForest(n_jobs=-1, random_state=42)

In [21]:
# Check the model's unusualness scores
val_scores = -val_model.score_samples(X_val)  # Higher = more unusual

score_summary = pd.DataFrame({
    "actual_label": y_val.to_numpy(),
    "unusualness": val_scores
})

display(
    score_summary.groupby("actual_label")["unusualness"]
    .agg(["median", "mean"])
    .round(3)
)

,median,mean
actual_label,,
0,0.394,0.408
1,0.458,0.477


In [22]:
# Try a more sensitive alarm
threshold = score_summary.loc[
    score_summary["actual_label"] == 0, "unusualness"
].quantile(0.90)

val_flags = (val_scores >= threshold).astype(int)

print("Alarm threshold:", round(threshold, 3))
display(pd.crosstab(
    y_val,
    val_flags,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["Flagged (0=no, 1=yes)"]
))

Alarm threshold: 0.479


"Flagged (0=no, 1=yes)",0,1
"Actual (0=normal, 1=attack)",,
0,12594,1406
1,17550,12286


In [23]:
# Compare with the no-reply rule
simple_val_flags = (X_val["dpkts"] == 0).astype(int)

display(pd.crosstab(
    y_val,
    simple_val_flags,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["No-reply rule (0=no, 1=yes)"]
))

"No-reply rule (0=no, 1=yes)",0,1
"Actual (0=normal, 1=attack)",,
0,12353,1647
1,10499,19337


In [24]:
# Train a supervised model
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print("Random Forest trained")

Random Forest trained


In [25]:
# Evaluate Random Forest on validation data
from sklearn.metrics import confusion_matrix, precision_score, recall_score

rf_val_flags = rf_model.predict(X_val)

display(pd.crosstab(
    y_val,
    rf_val_flags,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["Random Forest (0=no, 1=yes)"]
))

print("Precision:", round(precision_score(y_val, rf_val_flags), 3))
print("Recall:", round(recall_score(y_val, rf_val_flags), 3))

"Random Forest (0=no, 1=yes)",0,1
"Actual (0=normal, 1=attack)",,
0,12450,1550
1,494,29342


Precision: 0.95
Recall: 0.983


In [26]:
# Check the seperate test file
rf_test_flags = rf_model.predict(test_X)

display(pd.crosstab(
    test_y,
    rf_test_flags,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["Random Forest (0=no, 1=yes)"]
))

print("Test precision:", round(precision_score(test_y, rf_test_flags), 3))
print("Test recall:", round(recall_score(test_y, rf_test_flags), 3))

"Random Forest (0=no, 1=yes)",0,1
"Actual (0=normal, 1=attack)",,
0,25636,11364
1,405,44927


Test precision: 0.798
Test recall: 0.991


In [27]:
# Inspect the false alarm 
false_alarms = test_df.loc[
    (test_y == 0) & (rf_test_flags == 1)
]

print("False alarms:", len(false_alarms))
display(false_alarms["proto"].value_counts().head(10))

False alarms: 11364


proto
tcp     8421
udp     2927
igmp      16
Name: count, dtype: int64

In [28]:
# Compare false alarm rates by protocol
normal_test = test_df.loc[test_y == 0, ["proto"]].copy()
normal_test["flagged"] = rf_test_flags[test_y.to_numpy() == 0]

protocol_errors = (
    normal_test.groupby("proto")["flagged"]
    .agg(rows="size", false_alarm_rate="mean")
)

protocol_errors["false_alarm_rate"] *= 100
display(protocol_errors.sort_values("rows", ascending=False).head(10).round(1))

,rows,false_alarm_rate
proto,,
tcp,27848,30.2
udp,8097,36.1
arp,987,0.0
ospf,38,0.0
igmp,30,53.3


In [29]:
# Check false alarms by connection state
normal_test = test_df.loc[test_y == 0, ["state"]].copy()
normal_test["flagged"] = rf_test_flags[test_y.to_numpy() == 0]

state_errors = normal_test.groupby("state")["flagged"].agg(
    rows="size",
    false_alarm_rate="mean"
)
state_errors["false_alarm_rate"] *= 100

display(state_errors.sort_values("rows", ascending=False).round(1))

,rows,false_alarm_rate
state,,
FIN,24172,34.8
CON,6633,0.1
INT,4485,65.3
REQ,1707,0.9
ACC,2,0.0
RST,1,100.0


In [30]:
# Compare validation and test errors by state
val_normal = pd.DataFrame({
    "state": df.loc[X_val.index, "state"].to_numpy(),
    "actual": y_val.to_numpy(),
    "flagged": rf_val_flags
})

val_state_errors = (
    val_normal[val_normal["actual"] == 0]
    .groupby("state")["flagged"]
    .agg(rows="size", false_alarm_rate="mean")
)

val_state_errors["false_alarm_rate"] *= 100
display(val_state_errors.loc[["FIN", "INT"]].round(1))

,rows,false_alarm_rate
state,,
FIN,9267,12.4
INT,1430,28.2


In [31]:
# Compare normal FIN connections
val_fin = df.loc[X_val.index]
val_fin = val_fin[(val_fin["label"] == 0) & (val_fin["state"] == "FIN")]

test_fin = test_df[
    (test_df["label"] == 0) & (test_df["state"] == "FIN")
]

features = ["dur", "spkts", "dpkts", "sbytes", "dbytes"]

comparison = pd.DataFrame({
    "Validation median": val_fin[features].median(),
    "Test median": test_fin[features].median()
})

display(comparison.round(2))

,Validation median,Test median
dur,0.23,0.59
spkts,22.00,14.00
dpkts,18.00,12.00
sbytes,2516.00,1540.00
dbytes,3380.00,1268.00


In [32]:
# Count identical connection records
print("Identical clue rows:", X.duplicated().sum())
print("Total rows:", len(X))

Identical clue rows: 74301
Total rows: 175341


In [33]:
# Check overlap between training and validation
train_fingerprints = pd.util.hash_pandas_object(X_train, index=False)
val_fingerprints = pd.util.hash_pandas_object(X_val, index=False)

overlap = val_fingerprints.isin(set(train_fingerprints))
print("Validation rows also seen as identical clues in training:", overlap.sum())
print("Total validation rows:", len(X_val))

Validation rows also seen as identical clues in training: 20241
Total validation rows: 43836


In [34]:
# Make a duplicate-safe split
from sklearn.model_selection import StratifiedGroupKFold

groups = pd.util.hash_pandas_object(X, index=False)
splitter = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

train_idx, val_idx = next(splitter.split(X_encoded, y, groups))

X_train_g = X_encoded.iloc[train_idx]
X_val_g = X_encoded.iloc[val_idx]
y_train_g = y.iloc[train_idx]
y_val_g = y.iloc[val_idx]

print("Training:", X_train_g.shape)
print("Validation:", X_val_g.shape)
print("Validation labels:")
print(y_val_g.value_counts())

Training: (132175, 194)
Validation: (43166, 194)
Validation labels:
label
1    29129
0    14037
Name: count, dtype: int64


In [35]:
# Retrain on the duplicate-safe split
rf_grouped = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_grouped.fit(X_train_g, y_train_g)
print("Duplicate-safe model trained")

Duplicate-safe model trained


In [36]:
# Check the duplicate-safe validation result
grouped_flags = rf_grouped.predict(X_val_g)

display(pd.crosstab(
    y_val_g,
    grouped_flags,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["Predicted (0=normal, 1=attack)"]
))

print("Precision:", round(precision_score(y_val_g, grouped_flags), 3))
print("Recall:", round(recall_score(y_val_g, grouped_flags), 3))

"Predicted (0=normal, 1=attack)",0,1
"Actual (0=normal, 1=attack)",,
0,12200,1837
1,418,28711


Precision: 0.94
Recall: 0.986


In [37]:
# Inspect the model''s top clues
importance = pd.Series(
    rf_grouped.feature_importances_,
    index=X_encoded.columns
)

display(importance.sort_values(ascending=False).head(10).round(3))

ct_state_ttl    0.127
sttl            0.115
dload           0.059
rate            0.048
dttl            0.045
sload           0.040
dmean           0.037
dbytes          0.031
sbytes          0.030
tcprtt          0.030
dtype: float64

In [38]:
# Check whether the top clues changes
# Compare important clues in normal FIN connections
features = ["ct_state_ttl", "sttl"]

display(pd.DataFrame({
    "Validation normal FIN": val_fin[features].median(),
    "Test normal FIN": test_fin[features].median()
}))

,Validation normal FIN,Test normal FIN
ct_state_ttl,0.0,0.0
sttl,31.0,31.0


In [39]:
# Check one clue in more detail
# Compare the distribution of ct_state_ttl
display(pd.DataFrame({
    "Validation %": val_fin["ct_state_ttl"].value_counts(normalize=True).mul(100).round(1),
    "Test %": test_fin["ct_state_ttl"].value_counts(normalize=True).mul(100).round(1),
}).fillna(0).sort_index())

,Validation %,Test %
ct_state_ttl,,
0,77.1,50.1
1,22.9,49.9


In [40]:
# Check the false alarms
# False-alarm rate for each ct_state_ttl value
normal_test_fin = test_fin.copy()
normal_test_fin["flagged"] = test_flags[normal_test_fin.index]

display(
    normal_test_fin.groupby("ct_state_ttl")["flagged"]
    .agg(rows="size", false_alarm_rate="mean")
    .assign(false_alarm_rate=lambda table: (table["false_alarm_rate"] * 100).round(1))
)

,rows,false_alarm_rate
ct_state_ttl,,
0,12105,2.5
1,12067,6.0


In [41]:
# Check the prediction counts
# Check which predictions test_flags contains
print("All test flags:", pd.Series(test_flags).value_counts())
print("Normal FIN rows:", len(test_fin))
print("Flags matched to normal FIN:", len(test_flags[test_fin.index]))

All test flags: 0    62307
1    20025
Name: count, dtype: int64
Normal FIN rows: 24172
Flags matched to normal FIN: 24172


In [44]:
# Use the Random Forest Predictions
# False-alarm rate for the Random Forest, by ct_state_ttl
# Use the Random Forest predictions for the test rows
rf_test_flags = pd.Series(rf_model.predict(test_X), index=test_X.index)

normal_test_fin = test_fin.copy()
normal_test_fin["flagged"] = rf_test_flags.loc[normal_test_fin.index]

display(
    normal_test_fin.groupby("ct_state_ttl")["flagged"]
    .agg(rows="size", false_alarm_rate="mean")
    .assign(false_alarm_rate=lambda table: (table["false_alarm_rate"] * 100).round(1))
)

,rows,false_alarm_rate
ct_state_ttl,,
0,12105,0.0
1,12067,69.7


In [45]:
# Compare the validation
# Random Forest false alarms for normal FIN validation rows
val_mask = (y_val == 0) & (X.loc[X_val.index, "state"] == "FIN")

val_fin_check = X.loc[X_val.index[val_mask], ["ct_state_ttl"]].copy()
val_fin_check["flagged"] = rf_model.predict(X_val.loc[val_mask])

display(
    val_fin_check.groupby("ct_state_ttl")["flagged"]
    .agg(rows="size", false_alarm_rate="mean")
    .assign(false_alarm_rate=lambda table: (table["false_alarm_rate"] * 100).round(1))
)

,rows,false_alarm_rate
ct_state_ttl,,
0,7146,0.0
1,2121,54.0


In [46]:
# Experiment : Remove on clue
# Train the same model without ct_state_ttl
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score

columns_without_ttl_clue = X_train_g.columns.drop("ct_state_ttl")

rf_without_clue = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_without_clue.fit(
    X_train_g[columns_without_ttl_clue],
    y_train_g
)

new_val_flags = rf_without_clue.predict(
    X_val_g[columns_without_ttl_clue]
)

display(pd.crosstab(
    y_val_g,
    new_val_flags,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["New model (0=normal, 1=attack)"]
))

print("Precision:", round(precision_score(y_val_g, new_val_flags), 3))
print("Recall:", round(recall_score(y_val_g, new_val_flags), 3))

"New model (0=normal, 1=attack)",0,1
"Actual (0=normal, 1=attack)",,
0,12208,1829
1,406,28723


Precision: 0.94
Recall: 0.986


In [47]:
# Test the new model 
# Evaluate the model without ct_state_ttl on the official test set
new_test_flags = rf_without_clue.predict(
    test_X[columns_without_ttl_clue]
)

display(pd.crosstab(
    test_y,
    new_test_flags,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["New model (0=normal, 1=attack)"]
))

print("Precision:", round(precision_score(test_y, new_test_flags), 3))
print("Recall:", round(recall_score(test_y, new_test_flags), 3))

"New model (0=normal, 1=attack)",0,1
"Actual (0=normal, 1=attack)",,
0,25352,11648
1,331,45001


Precision: 0.794
Recall: 0.993


In [48]:
# Fair comparison: grouped model on the test set 
# Test the grouped model that still has ct_state_ttl
grouped_test_flags = rf_grouped.predict(test_X)

display(pd.crosstab(
    test_y,
    grouped_test_flags,
    rownames=["Actual (0=normal, 1=attack)"],
    colnames=["Grouped model (0=normal, 1=attack)"]
))

print("Precision:", round(precision_score(test_y, grouped_test_flags), 3))
print("Recall:", round(recall_score(test_y, grouped_test_flags), 3))

"Grouped model (0=normal, 1=attack)",0,1
"Actual (0=normal, 1=attack)",,
0,25456,11544
1,353,44979


Precision: 0.796
Recall: 0.992


## What we learned from the models
The Random Forest detected most attacks in the official test set, but it also flagged many normal connections. The grouped model had 99.2% recall and 79.6% precision, with 11,544 false alarms and 353 missed attacks.
Normal FIN connections with ct_state_ttl = 1 were especially likely to be flagged. Removing that feature did not meaningfully reduce false alarms, so we kept the grouped model.
Conclusion: This is a useful experiment in cyber threat detection, but the current model should not be treated as a reliable real-world security alarm. Too many normal connections would require investigation.

In [49]:
# Find a safer warning threshold
# Compare warning thresholds on grouped validation data
from sklearn.metrics import confusion_matrix

attack_chance = rf_grouped.predict_proba(X_val_g)[:, 1]

for threshold in [0.5, 0.7, 0.9]:
    warnings = (attack_chance >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val_g, warnings).ravel()

    print(
        f"Threshold {threshold}: "
        f"{fp} false alarms, {fn} missed attacks, "
        f"{tp} attacks caught"
    )

Threshold 0.5: 1837 false alarms, 418 missed attacks, 28711 attacks caught
Threshold 0.7: 646 false alarms, 1868 missed attacks, 27261 attacks caught
Threshold 0.9: 55 false alarms, 4944 missed attacks, 24185 attacks caught


In [50]:
# Train a model that pays extra attention to normal connections
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

rf_fewer_alarms = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight={0: 2, 1: 1},
    random_state=42,
    n_jobs=-1
)

rf_fewer_alarms.fit(X_train_g, y_train_g)
new_flags = rf_fewer_alarms.predict(X_val_g)

tn, fp, fn, tp = confusion_matrix(y_val_g, new_flags).ravel()

print("False alarms:", fp)
print("Missed attacks:", fn)
print("Attacks caught:", tp)

False alarms: 846
Missed attacks: 1507
Attacks caught: 27622


In [51]:
# Check both kinds of mistakes on official test data
weighted_flags = rf_fewer_alarms.predict(test_X)

tn, fp, fn, tp = confusion_matrix(test_y, weighted_flags).ravel()

print("False alarms:", fp)
print("Missed attacks:", fn)
print("Attacks caught:", tp)

False alarms: 6378
Missed attacks: 1288
Attacks caught: 44044


In [52]:
# Train XGBoost on the same grouped training data
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_g, y_train_g)

print("XGBoost training complete.")

XGBoost training complete.


In [53]:
# Count false alarms and missed attacks
from sklearn.metrics import confusion_matrix, precision_score, recall_score

xgb_val_flags = xgb_model.predict(X_val_g)
tn, fp, fn, tp = confusion_matrix(y_val_g, xgb_val_flags).ravel()

print("False alarms:", fp)
print("Missed attacks:", fn)
print("Attacks caught:", tp)
print("Precision:", round(precision_score(y_val_g, xgb_val_flags), 3))
print("Recall:", round(recall_score(y_val_g, xgb_val_flags), 3))

False alarms: 1295
Missed attacks: 686
Attacks caught: 28443
Precision: 0.956
Recall: 0.976


In [54]:
# Warn if Random Forest OR XGBoost flags a connection
rf_val_flags = rf_grouped.predict(X_val_g)
combined_flags = ((rf_val_flags == 1) | (xgb_val_flags == 1)).astype(int)

tn, fp, fn, tp = confusion_matrix(y_val_g, combined_flags).ravel()

print("False alarms:", fp)
print("Missed attacks:", fn)
print("Attacks caught:", tp)

False alarms: 1896
Missed attacks: 345
Attacks caught: 28784


In [55]:
# Test the combined warning
# Warn when either grouped model flags a test connection
rf_test = rf_grouped.predict(test_X)
xgb_test = xgb_model.predict(test_X)
combined_test = ((rf_test == 1) | (xgb_test == 1)).astype(int)

tn, fp, fn, tp = confusion_matrix(test_y, combined_test).ravel()

print("False alarms:", fp)
print("Missed attacks:", fn)
print("Attacks caught:", tp)

False alarms: 12176
Missed attacks: 242
Attacks caught: 45090
